# Data Analysis Agent

> Ask questions about your data in plain English. The agent writes the code, runs it, draws the charts, and explains the results.

## The Problem

Most people who own data cannot query it. A shop owner has a sales CSV, a clinic has patient visit records, a farmer's cooperative has yield spreadsheets, but answering a simple question like *"Which products sold worst last quarter?"* still means finding someone who knows pandas or SQL.

This project builds an **AI agent** that closes that gap.

## What It Does

You load a dataset and ask a question. The agent then works in a loop:

1. **Reads** a summary of your data (columns, types, sample rows)
2. **Writes** Python (pandas, matplotlib) to answer the question
3. **Runs** the code and looks at the output
4. **Fixes** its own mistakes if the code errors
5. **Explains** the result in plain English, based only on what the code actually printed

Example questions:

- "Which 5 products sold worst last quarter?"
- "Plot monthly revenue and tell me if there's a trend."
- "Are there missing values or outliers I should worry about?"
- "Now break that down by region." *(follow-ups keep context)*

## Why an Agent, Not Just a Chatbot?

A chatbot guesses at answers from its training data. This agent **does the work**: it computes on your real data, so every number in its answer is traceable to code you can read. If its first attempt fails (a wrong column name, a bad date format), it reads the error and tries again, which is what makes it an agent rather than a one-shot text generator.

## How This Notebook Is Organized

This is not a flat pile of cells. The notebook is structured as a **production-grade project in seven layers**, each with one job. Some layers are code you run; others use `%%writefile` cells to generate the real project files on disk, so running the notebook top to bottom produces a proper repository.

| # | Layer | Job | What it produces |
|---|-------|-----|------------------|
| 1 | **Foundation** | Project identity, dependencies, docs | Folder structure, `requirements.txt`, `README.md`, `LICENSE` |
| 2 | **Configuration** | Environment-specific secrets and settings | `.env.example`, `.gitignore`, `config.py` |
| 3 | **Source** | The actual agent logic | `tools.py`, `prompts.py`, `agent.py`, `main.py` |
| 4 | **Containerization** | Ship and run anywhere | `Dockerfile`, `docker-compose.yml` |
| 5 | **Observability** | See what it is doing | Structured JSON logs of every question, step, error, and token count |
| 6 | **Evaluation** | Prove it works, before and after every change | Test questions with known answers, unit tests |
| 7 | **Tooling & Automation** | CI/CD, editor config, scripts | GitHub Actions workflow, `Makefile`, `.editorconfig` |

## Tech Stack

- **Language:** Python 3.12
- **Model:** Llama 3.3 70B via the [Groq](https://console.groq.com/keys) API (free tier available, supports tool calling)
- **Data:** pandas, matplotlib (DuckDB planned for SQL)
- **Config:** python-dotenv, pydantic-settings
- **Quality:** pytest, GitHub Actions
- **Packaging:** Docker

The model is swappable. Changing the client and model name in Layer 2 and Layer 3 is enough to move to Claude, OpenAI, or a local model.

## Before You Start

1. Create a free API key at **console.groq.com/keys**
2. Have a CSV ready to analyze (any tabular data works)
3. Python 3.10+ and Jupyter installed

## A Note on Safety

The agent executes code that the model writes. In a personal notebook with data you trust, that is fine, but **`exec()` is not a sandbox**: the code runs with the same access as your Python session. Layer 4 (Docker) is what contains it. Do not expose this agent to other users without running it inside a container or a dedicated sandbox.

## Roadmap

- [ ] Add a SQL tool (DuckDB) so the agent can choose between pandas and SQL
- [ ] File upload widget for non-technical users
- [ ] A data-cleaning tool that describes its changes before applying them
- [ ] Larger evaluation set with real expected answers
- [ ] Hosted observability (Langfuse or LangSmith)

## Author

*Willard Odongo*

## Layer 1: Foundation

This is the project’s identity — the stuff that answers “what is this, who owns it, how do I install it.”

In [1]:
from pathlib import Path
for d in ["src/agent", "tests", "evals", "data", "logs", ".github/worklfows"]:
    Path(d).mkdir(parents=True, exist_ok=True)
Path("src/agent/__init__.py").touch()

In [2]:
%%writefile requirements.txt
pandas
matplotlib
groq
python-dotenv
pydantic-settings
duckdb
pytest

Overwriting requirements.txt


In [3]:
%%writefile LICENSE
MIT License. Willard Odongo, 2026.

Overwriting LICENSE


## Layer 2: Configuration

AI agents talk to APIs — model providers, vector stores, external tools. Each environment needs its own keys and settings, and none of them belong in source code.

In [4]:
%%writefile .env.example
GROQ_API_KEY=your_api_key_here
MODEL=openai/gpt-oss-120b
MAX_STEPS=6
LOG_LEVEL=INFO

Overwriting .env.example


In [5]:
%%writefile .gitignore
.env
logs/
data/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


In [6]:
%%writefile src/agent/config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    groq_api_key: str
    model: str = "openai/gpt-oss-120b"
    max_steps: int = 6
    log_level: str = "INFO"

    class Config:
        env_file = ".env"

settings = Settings()

Overwriting src/agent/config.py


## Layer 3:Source

`src/` — the actual agent: graph/workflow definitions, tools, prompts, retrieval logic, API layer. Everything that makes the agent an agent lives here, cleanly separated from infra and tooling.

In [7]:
%%writefile src/agent/tools.py
import io, contextlib, traceback
import pandas as pd
import matplotlib
matplotlib.use("Agg")           # headless-safe; charts are saved, not shown
import matplotlib.pyplot as plt

def run_python(code: str, df: pd.DataFrame) -> str:
    out = io.StringIO()
    env = {"df": df, "pd": pd, "plt": plt}
    try:
        with contextlib.redirect_stdout(out):
            exec(code, env)
        plt.savefig("logs/last_chart.png") if plt.get_fignums() else None
        plt.close("all")
    except Exception:
        return out.getvalue() + "\nERROR:\n" + traceback.format_exc(limit=2)
    return out.getvalue() or "(ran fine, nothing printed, use print())"

TOOLS = [{
    "type": "function",
    "function": {
        "name": "run_python",
        "description": "Run Python against the DataFrame `df`. Returns printed output or an error.",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string"}},
            "required": ["code"],
        },
    },
}]

Overwriting src/agent/tools.py


In [8]:
%%writefile src/agent/prompts.py
import io

def describe_df(df):
    buf = io.StringIO()
    df.info(buf=buf)
    return f"{buf.getvalue()}\nSample rows:\n{df.head(3).to_string()}"

def build_system_prompt(df):
    return f"""You are a data analyst. A pandas DataFrame named `df` is loaded.
Answer by writing Python and calling run_python.
- Use print() to show results; you only see printed output.
- Charts: matplotlib with titles and axis labels.
- If code errors, read the error and fix it.
- Never invent numbers. Base answers only on tool output.
- End with a short plain-English explanation.

DataFrame info:
{describe_df(df)}"""

Overwriting src/agent/prompts.py


In [9]:
%%writefile src/agent/agent.py
import json, time
from groq import Groq
from .config import settings
from .tools import TOOLS, run_python
from .prompts import build_system_prompt
from .observability import get_logger

log = get_logger()
client = Groq(api_key=settings.groq_api_key)

def ask(question, df, messages=None):
    messages = messages or [{"role": "system", "content": build_system_prompt(df)}]
    messages.append({"role": "user", "content": question})
    trace = {"question": question, "steps": []}

    for step in range(settings.max_steps):
        t0 = time.time()
        resp = client.chat.completions.create(
            model=settings.model, messages=messages, tools=TOOLS, tool_choice="auto"
        )
        msg = resp.choices[0].message
        latency = round(time.time() - t0, 2)

        if not msg.tool_calls:
            trace.update(answer=msg.content, tokens=resp.usage.total_tokens)
            log.info(json.dumps(trace))
            return msg.content, messages

        messages.append({
            "role": "assistant", "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls],
        })
        for tc in msg.tool_calls:
            code = json.loads(tc.function.arguments)["code"]
            result = run_python(code, df)
            trace["steps"].append({"code": code, "result": result[:500],
                                   "latency_s": latency, "error": "ERROR" in result})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result[:4000]})

    trace["answer"] = None
    log.warning(json.dumps(trace))
    return "Stopped: hit step limit.", messages

Overwriting src/agent/agent.py


In [10]:
%%writefile src/agent/main.py
import sys
import pandas as pd
from .agent import ask

if __name__ == "__main__":
    df = pd.read_csv(sys.argv[1])
    answer, _ = ask(sys.argv[2], df)
    print(answer)

Overwriting src/agent/main.py


## Layer 4: Containerization

`docker/` — Dockerfiles, docker-compose configs, entrypoint scripts. This is what makes "works on my machine" irrelevant — the agent runs identically in dev, staging, and prod.

In [11]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY src/ src/
ENV PYTHONPATH=/app/src
ENTRYPOINT ["python", "-m", "agent.main"]

Overwriting Dockerfile


In [12]:
%%writefile docker-compose.yml
services:
  agent:
    build: .
    env_file: .env
    volumes:
      - ./data:/app/data:ro
      - ./logs:/app/logs

Overwriting docker-compose.yml


## Layer 5: Observability

If you can’t see what your agent is doing in production, you don’t have a production system — you have a black box you’re hoping doesn’t break.

In [13]:
%%writefile src/agent/observability.py
import logging, os

def get_logger():
    os.makedirs("logs", exist_ok=True)
    logger = logging.getLogger("agent")
    if not logger.handlers:
        logger.setLevel(os.getenv("LOG_LEVEL", "INFO"))
        logger.addHandler(logging.FileHandler("logs/agent.jsonl"))
        logger.addHandler(logging.StreamHandler())
    return logger

Overwriting src/agent/observability.py


In [14]:
from pathlib import Path
import pandas as pd

log_path = Path("logs/agent.jsonl")
if not log_path.exists() or log_path.stat().st_size == 0:
    print("No agent runs logged yet - run a question first.")
else:
    pd.read_json(log_path, lines=True).tail()

## Layer 6: Evaluation

`evals/` — this is what separates a serious AI project from a toy. Automated evaluation sets, golden datasets, scoring scripts, regression checks that run whenever the prompt, model, or logic changes. Without this layer, every change to the agent is a guess.

In [15]:
%%writefile evals/cases.json
[
  {"question": "How many rows are in the data?", "expected": "100"},
  {"question": "What is the total revenue?", "expected": "46269.26"}
]

Overwriting evals/cases.json


In [16]:
%%writefile evals/run_evals.py
import json, sys
import pandas as pd
sys.path.insert(0, "src")
from agent.agent import ask

df = pd.read_csv(sys.argv[1])
cases = json.load(open("evals/cases.json"))
passed = 0
for c in cases:
    answer, _ = ask(c["question"], df)
    ok = c["expected"] in (answer or "").replace(",", "")
    passed += ok
    print(("PASS" if ok else "FAIL"), "-", c["question"])
print(f"\n{passed}/{len(cases)} passed")

Overwriting evals/run_evals.py


In [17]:
%%writefile tests/test_tools.py
import sys; sys.path.insert(0, "src")
import pandas as pd
from agent.tools import run_python

def test_runs_code():
    df = pd.DataFrame({"a": [1, 2, 3]})
    assert "6" in run_python("print(df.a.sum())", df)

def test_reports_errors():
    df = pd.DataFrame({"a": [1]})
    assert "ERROR" in run_python("print(df.missing)", df)

Overwriting tests/test_tools.py


## Layer 7: Tooling & Automation

`.github/` — CI/CD workflows: linting, tests, evals, and deployment pipelines running automatically on every push/PR

In [18]:
%%writefile .github/workflows/ci.yml
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12" }
      - run: pip install -r requirements.txt
      - run: pytest tests/

Overwriting .github/workflows/ci.yml


In [19]:
%%writefile Makefile
PY ?= .venv/bin/python
export PYTHONPATH := src

.PHONY: install test eval run docker

install:
	$(PY) -m pip install -r requirements.txt
test:
	$(PY) -m pytest tests/ -q
eval:
	$(PY) evals/run_evals.py data/sales.csv
run:
	$(PY) -m agent.main data/sales.csv "$(Q)"
docker:
	docker compose run --rm agent data/sales.csv "$(Q)"

Overwriting Makefile


In [20]:
%%writefile .editorconfig
root = true
[*]
indent_style = space
indent_size = 4
end_of_line = lf
insert_final_newline = true

Overwriting .editorconfig


In [21]:
import sys; sys.path.insert(0, "src")
import pandas as pd
from agent.agent import ask

df = pd.read_csv("data/sales.csv")
answer, history = ask("Which 5 products sold worst last quarter?", df)
print(answer)
answer, history = ask("Now break that down by region", df, history)   # follow-up keeps context

{"question": "Which 5 products sold worst last quarter?", "steps": [{"code": "import pandas as pd\n# ensure date is datetime\ndf['date'] = pd.to_datetime(df['date'])\nmax_date = df['date'].max()\n# define last quarter as 3 months before max_date (inclusive)\nstart_date = max_date - pd.DateOffset(months=3) + pd.DateOffset(days=1)  # start after 3 months ago\nmask = (df['date'] >= start_date) & (df['date'] <= max_date)\nlast_q = df.loc[mask]\n# aggregate quantity by product\nproduct_sales = last_q.groupby('product')['quantity'].sum().reset_index()\n# sort ascending (worst sales)\nworst5 = product_sales.nsmallest(5, 'quantity')\nprint('Max date:', max_date)\nprint('Start date:', start_date)\nprint(worst5)", "result": "Max date: 2026-11-14 00:00:00\nStart date: 2026-08-15 00:00:00\n               product  quantity\n5        Standing Desk         6\n1    Bluetooth Speaker        13\n0         27in Monitor        24\n3  Mechanical Keyboard        31\n2         Laptop Stand        34\n", "lat

**5 products with the lowest total quantity sold in the last quarter (15 Aug 2026 – 14 Nov 2026)**  

| Rank | Product               | Total Quantity Sold |
|------|-----------------------|----------------------|
| 1    | Standing Desk         | 6 |
| 2    | Bluetooth Speaker     | 13 |
| 3    | 27in Monitor          | 24 |
| 4    | Mechanical Keyboard   | 31 |
| 5    | Laptop Stand          | 34 |

These are the products that sold the fewest units during the most recent three‑month period in the dataset.


{"question": "Now break that down by region", "steps": [{"code": "# Use same last quarter mask as before\nimport pandas as pd\ndf['date'] = pd.to_datetime(df['date'])\nmax_date = df['date'].max()\nstart_date = max_date - pd.DateOffset(months=3) + pd.DateOffset(days=1)\nmask = (df['date'] >= start_date) & (df['date'] <= max_date)\nlast_q = df.loc[mask]\n# worst 5 products from previous result\nproduct_sales = last_q.groupby('product')['quantity'].sum().reset_index()\nworst5 = product_sales.nsmallest(5, 'quantity')['product'].tolist()\n# filter last_q to those products\nfiltered = last_q[last_q['product'].isin(worst5)]\n# group by product and region, sum quantity\nbreakdown = filtered.groupby(['product', 'region'])['quantity'].sum().reset_index()\nprint('Worst 5 products:', worst5)\nprint(breakdown)", "result": "Worst 5 products: ['Standing Desk', 'Bluetooth Speaker', '27in Monitor', 'Mechanical Keyboard', 'Laptop Stand']\n                product region  quantity\n0          27in Monitor